<a href="https://colab.research.google.com/github/mugalan/gsuite-based-attendance-monitoring/blob/main/QR_code_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#QR Code Generator

In [ ]:
import qrcode
import hashlib
import time
from IPython.display import display, Image, clear_output

# CONFIGURATION
WEB_APP_URL = "YOUR_APP_SCRIP_URL"
SECRET_PASSWORD = "123"
# MUST match the Script password

def get_security_key():
    # Syncs with the 5-second window in the Apps Script
    time_slot = int(time.time() // 300)
    raw_str = f"{SECRET_PASSWORD}{time_slot}"
    return hashlib.md5(raw_str.encode()).hexdigest()

try:
    while True:
        clear_output(wait=True)

        # Create the secure link
        secure_key = get_security_key()
        secure_url = f"{WEB_APP_URL}?key={secure_key}"

        # Generate QR
        qr = qrcode.make(secure_url)
        qr.save("secure_qr.png")

        print("🛡️ SECURE ATTENDANCE SYSTEM ACTIVE")
        print(f"Time: {time.strftime('%H:%M:%S')} | Key: {secure_key[:8]}...")
        print("Note: This QR expires every 5 seconds to prevent cheating.")

        display(Image("secure_qr.png"))
        time.sleep(300)
except KeyboardInterrupt:
    print("System Offline.")

#Google Sheet App Script

In [ ]:
// Replace YOUR_SECRET_PASSWORD with any word you like
var SECRET_PASSWORD = "123";

function doGet(e) {
  var userKey = e.parameter.key;
  var isValid = checkKey(userKey);

  var html = '<html><body style="font-family: sans-serif; text-align:center; padding:50px;">';

  if (!isValid) {
    html += '<h1 style="color:red;">❌ Invalid or Expired QR</h1>';
    html += '<p>Please scan the live QR code displayed in the classroom.</p>';
  } else {
    html += '<div id="main"><h2>Verify Attendance</h2>' +
            '<button style="padding:15px 30px; font-size:18px; background:#1a73e8; color:white; border:none; border-radius:5px;" onclick="record()">Confirm My Identity</button></div>' +
            '<script>' +
            '  function record() {' +
            '    google.script.run.withSuccessHandler(done).saveEmail("' + userKey + '");' +
            '  }' +
            '  function done(info) {' +
            '    document.body.innerHTML = "<h1>✅ Success!</h1><p>Thanks, " + info.name + "!<br>Position: #" + info.count + "</p>";' +
            '  }' +
            '</script>';
  }
  html += '</body></html>';
  return HtmlService.createHtmlOutput(html);
}

function checkKey(key) {
  var now = new Date();
  var timeSlot = Math.floor(now.getTime() / 300000); // 5-second window
  // This generates a code that only matches the current 5-second window
  var expectedKey = Utilities.computeDigest(Utilities.DigestAlgorithm.MD5, SECRET_PASSWORD + timeSlot);
  var expectedKeyStr = expectedKey.map(function(byt){return ('0' + (byt & 0xFF).toString(16)).slice(-2)}).join('');

  return key === expectedKeyStr;
}

function saveEmail(key) {
  if (!checkKey(key)) throw new Error("Security Timeout: Please re-scan");

  var ss = SpreadsheetApp.getActiveSpreadsheet();
  var sheetName = Utilities.formatDate(new Date(), "GMT+5:30", "dd-MM-yyyy");
  var sheet = ss.getSheetByName(sheetName) || ss.insertSheet(sheetName, 0);

  var user = Session.getActiveUser();
  var email = user.getEmail();
  var name = email.split('@')[0];
  var timeString = Utilities.formatDate(new Date(), "GMT+5:30", "HH:mm:ss");

  sheet.appendRow([timeString, email, name]);
  return { name: name, count: (sheet.getLastRow() - 1), time: timeString };
}